# BÁO CÁO TIỀN XỬ LÝ DỮ LIỆU TĂNG TRƯỞNG TRẺ EM (NHANES & WHO)
Đây là notebook thực hiện quá trình tiền xử lý dữ liệu từ dữ liệu thô (raw data) của khảo sát NHANES (Hoa Kỳ) kết hợp với các bảng chuẩn tăng trưởng của Tổ chức Y tế Thế giới (WHO) để chuẩn bị tập dữ liệu huấn luyện cho mô hình học máy.

### Quy trình xử lý gồm:
1. Kết nối Google Drive và khai báo các đường dẫn thư mục.
2. Đọc và ghép nối (merge) các file nhân khẩu học (`DEMO`) và số đo thể chất (`BMX`) thô (định dạng `.XPT`) của từng chu kỳ khảo sát NHANES.
3. Lọc trẻ em dưới 24 tháng tuổi.
4. Tính toán các chỉ số tăng trưởng chuẩn (Z-Scores) và Bách phân vị (Percentiles) theo độ tuổi, giới tính và chiều cao/cân nặng sử dụng dữ liệu tham chiếu WHO.
5. Lưu tập dữ liệu đã làm sạch và dán nhãn thành công dưới dạng `.csv` để huấn luyện.

## Bước 1: Kết nối Google Drive và cấu hình đường dẫn

In [ ]:
# Kết nối Google Drive để truy cập dữ liệu
from google.colab import drive
import os

drive.mount('/content/drive')

# Đường dẫn thư mục chính lưu trữ project trên Drive của bạn
# Mặc định là thư mục DACN nằm ngoài My Drive của bạn
DRIVE_BASE_DIR = '/content/drive/MyDrive/DACN'

RAW_NHANES_DIR = os.path.join(DRIVE_BASE_DIR, 'data/raw')
WHO_REF_DIR = os.path.join(DRIVE_BASE_DIR, 'data/who_ref')
PROCESSED_DIR = os.path.join(DRIVE_BASE_DIR, 'data/processed')

# Khởi tạo các thư mục nếu chưa tồn tại
os.makedirs(RAW_NHANES_DIR, exist_ok=True)
os.makedirs(WHO_REF_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Cấu hình đường dẫn thư mục:")
print(f"- Thư mục chứa dữ liệu NHANES thô: {RAW_NHANES_DIR}")
print(f"- Thư mục chứa dữ liệu WHO tham chiếu: {WHO_REF_DIR}")
print(f"- Thư mục đầu ra của dữ liệu xử lý: {PROCESSED_DIR}")

## Bước 2: Khai báo thư viện và cài đặt các dependencies cần thiết

In [ ]:
# Cài đặt thư viện đọc Excel xlsx và thư viện SAS XPT
!pip install -q openpyxl pyreadstat

import pandas as pd
import numpy as np
import glob
import warnings
from scipy.special import erf
from math import sqrt

warnings.filterwarnings('ignore')
print("Đã import thành công các thư viện cần thiết!")

## Bước 3: Định nghĩa bộ công cụ tính toán Z-Score & Percentile theo chuẩn WHO

In [ ]:
def percentile_from_zscore(zscore):
    """Tính toán bách phân vị dựa trên Z-Score sử dụng phân phối Gaussian tích lũy."""
    if zscore is None or pd.isna(zscore):
        return np.nan
    z_value = float(zscore)
    return 100.0 * 0.5 * (1.0 + erf(z_value / sqrt(2.0)))

def zscore_from_lms(measurement, l, m, s):
    """Tính Z-Score sử dụng phương pháp LMS (Lambda, Mu, Sigma) của WHO."""
    if measurement is None or pd.isna(measurement) or pd.isna(m) or pd.isna(s):
        return np.nan
    
    y = float(measurement)
    median = float(m)
    scale = float(s)
    shape = float(l) if l is not None and not pd.isna(l) else 0.0
    
    if y <= 0 or median <= 0 or scale <= 0:
        return np.nan
    
    if abs(shape) < 1e-8:
        return float(np.log(y / median) / scale)
    return float((((y / median) ** shape) - 1.0) / (shape * scale))

def classification_from_zscore(zscore):
    """Phân loại trạng thái dinh dưỡng dựa trên thang đo Z-Score của WHO."""
    if zscore is None or pd.isna(zscore):
        return "unknown"
    value = float(zscore)
    if value < -3:
        return "severe_thin"
    if value < -2:
        return "thin"
    if value <= 1:
        return "normal"
    if value <= 2:
        return "overweight"
    return "obese"

def find_who_workbook(folder_name, sex, token):
    """Tìm file Excel chứa bảng tham chiếu WHO trong thư mục."""
    folder = os.path.join(WHO_REF_DIR, folder_name)
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Thư mục WHO không tồn tại: {folder}")
    
    xlsx_files = glob.glob(os.path.join(folder, "**/*.xlsx"), recursive=True)
    matches = [
        f for f in xlsx_files
        if sex in os.path.basename(f).lower() 
        and token in os.path.basename(f).lower() 
        and "zscore" in os.path.basename(f).lower()
    ]
    if not matches:
        raise FileNotFoundError(f"Không tìm thấy bảng Excel WHO phù hợp cho {folder_name} / {sex} / {token}")
    return sorted(matches)[0]

def load_reference(kind, sex):
    """Tải bảng tham chiếu LMS của WHO từ file Excel tương ứng."""
    kind = kind.lower().strip()
    sex = sex.lower().strip()
    
    if kind == "wfl":
        workbook = find_who_workbook("Weight-for-length_height", sex, "wfl")
    elif kind == "bfa":
        workbook = find_who_workbook("Body mass index-for-age (BMI-for-age)", sex, "acfa")
    elif kind == "wfa":
        workbook = find_who_workbook("Weight-for-age", sex, "wfa")
    elif kind == "lfa":
        workbook = find_who_workbook("Length-height-for-age", sex, "lhfa")
    else:
        raise ValueError(f"Loại đo chuẩn WHO không được hỗ trợ: {kind}")
        
    table = pd.read_excel(workbook, sheet_name=0)
    table.columns = [str(col).strip() for col in table.columns]
    
    x_col = next((c for c in table.columns if c.lower() in {"day", "age", "height", "length"}), None)
    if x_col is None:
        raise ValueError(f"Không thể xác định cột trục hoành (X-axis) trong file Excel: {workbook}")
        
    for col in [x_col, "L", "M", "S"]:
        table[col] = pd.to_numeric(table[col], errors="coerce")
    table = table.dropna(subset=[x_col, "L", "M", "S"]).sort_values(x_col).reset_index(drop=True)
    return x_col, table

def interpolate_lms(x_col, table, x_value):
    """Nội suy tuyến tính các tham số L, M, S từ bảng chuẩn WHO cho giá trị x cụ thể."""
    x_series = table[x_col].astype(float)
    x_val_f = float(x_value)
    
    if x_val_f <= float(x_series.iloc[0]):
        return table.iloc[0]
    if x_val_f >= float(x_series.iloc[-1]):
        return table.iloc[-1]
        
    interpolated = {}
    for col in ["L", "M", "S"]:
        interpolated[col] = float(np.interp(x_val_f, x_series, table[col].astype(float)))
    return pd.Series(interpolated)

def compute_who_score(kind, sex, x_value, measurement):
    """Tính toán Z-Score và Percentile."""
    x_col, table = load_reference(kind, sex)
    row = interpolate_lms(x_col, table, float(x_value))
    zscore = zscore_from_lms(measurement, row["L"], row["M"], row["S"])
    percentile = percentile_from_zscore(zscore)
    return zscore, percentile

def classify_row(row):
    """Dựa trên giới tính, ngày tuổi, chiều cao và cân nặng để đối chiếu chuẩn WHO phù hợp nhất."""
    sex_val = str(row.get("sex", "")).strip().lower()
    sex = "boys" if sex_val in {"1", "male", "boy", "m", "nam"} else "girls"
    
    age_months = pd.to_numeric(row.get("age_months"), errors="coerce")
    age_days = pd.to_numeric(row.get("age_days"), errors="coerce")
    weight_kg = pd.to_numeric(row.get("weight_kg"), errors="coerce")
    height_cm = pd.to_numeric(row.get("height_cm"), errors="coerce")
    bmi = pd.to_numeric(row.get("bmi"), errors="coerce")
    
    # 1. Trọng lượng theo độ tuổi (wfa)
    if pd.notna(age_days) and pd.notna(weight_kg):
        try:
            z, p = compute_who_score("wfa", sex, float(age_days), float(weight_kg))
            return pd.Series({"who_metric": "wfa", "who_zscore": z, "who_percentile": p, "who_class": classification_from_zscore(z)})
        except:
            pass
            
    # 2. Chiều dài theo độ tuổi (lfa)
    if pd.notna(age_days) and pd.notna(height_cm):
        try:
            z, p = compute_who_score("lfa", sex, float(age_days), float(height_cm))
            return pd.Series({"who_metric": "lfa", "who_zscore": z, "who_percentile": p, "who_class": classification_from_zscore(z)})
        except:
            pass
            
    # 3. Trọng lượng theo chiều dài/cao (wfl)
    if pd.notna(height_cm) and pd.notna(weight_kg):
        try:
            if pd.notna(age_months) and float(age_months) < 24:
                z, p = compute_who_score("wfl", sex, float(height_cm), float(weight_kg))
                return pd.Series({"who_metric": "wfl", "who_zscore": z, "who_percentile": p, "who_class": classification_from_zscore(z)})
            elif pd.notna(age_months) and float(age_months) <= 60 and pd.notna(bmi):
                z, p = compute_who_score("bfa", sex, float(age_days), float(bmi))
                return pd.Series({"who_metric": "bfa", "who_zscore": z, "who_percentile": p, "who_class": classification_from_zscore(z)})
        except:
            pass
            
    return pd.Series({"who_metric": np.nan, "who_zscore": np.nan, "who_percentile": np.nan, "who_class": "unknown"})

## Bước 4: Đọc và gộp dữ liệu thô (DEMO & BMX) từ NHANES

In [ ]:
# Hàm chuẩn hóa cấu trúc cột dữ liệu thô
def detect_column(cols, patterns):
    # Đảm bảo chuyển cột thành string trước khi upper (đề phòng pandas read_sas trả về bytes)
    uc = [str(c, encoding='utf-8').upper() if isinstance(c, bytes) else str(c).upper() for c in cols]
    for pat in patterns:
        for i, c in enumerate(uc):
            if pat in c:
                return cols[i]
    return None

def normalize_demo(df):
    seq = detect_column(df.columns, ['SEQN'])
    if seq is None: raise RuntimeError('Không tìm thấy cột SEQN trong file DEMO')
    df = df.rename(columns={seq: 'SEQN'})
    
    age_m_col = detect_column(df.columns, ['RIDAGEMN', 'RIDAGE_MN', 'RIDAGEM'])
    age_y_col = detect_column(df.columns, ['RIDAGEYR', 'RIDAGEY'])
    if age_m_col:
        df['age_months'] = pd.to_numeric(df[age_m_col], errors='coerce')
    elif age_y_col:
        df['age_months'] = pd.to_numeric(df[age_y_col], errors='coerce') * 12
    else:
        df['age_months'] = pd.NA
        
    sex_col = detect_column(df.columns, ['RIAGENDR', 'SEX', 'RIAGEND'])
    if sex_col:
        # Hỗ trợ giải mã nếu giá trị trả về dạng bytes
        df['sex_str'] = df[sex_col].map(lambda x: str(x, encoding='utf-8') if isinstance(x, bytes) else str(x))
        df['sex'] = df['sex_str'].map({'1': 'male', '2': 'female', '1.0': 'male', '2.0': 'female', 'male': 'male', 'female': 'female'}).fillna('unknown')
    else:
        df['sex'] = pd.NA
    return df[['SEQN', 'age_months', 'sex']]

def normalize_bmx(df):
    seq = detect_column(df.columns, ['SEQN'])
    if seq is None: raise RuntimeError('Không tìm thấy cột SEQN trong file BMX')
    df = df.rename(columns={seq: 'SEQN'})
    
    weight_col = detect_column(df.columns, ['BMXWT', 'WT', 'WEIGHT', 'WTKG'])
    # BMXRECUM là chiều dài khi nằm (cho bé dưới 24 tháng), BMXHT là chiều cao khi đứng
    recum_col = detect_column(df.columns, ['BMXRECUM', 'RECUM'])
    height_col = detect_column(df.columns, ['BMXHT', 'HT', 'HEIGHT', 'LENGTH'])
    
    if weight_col:
        df['weight_kg'] = pd.to_numeric(df[weight_col], errors='coerce')
    else:
        df['weight_kg'] = pd.NA
        
    # Kết hợp: Ưu tiên chiều dài nằm cho bé dưới 2 tuổi (nếu có), không thì dùng chiều cao đứng
    if recum_col and height_col:
        df['height_cm'] = pd.to_numeric(df[recum_col], errors='coerce').fillna(pd.to_numeric(df[height_col], errors='coerce'))
    elif recum_col:
        df['height_cm'] = pd.to_numeric(df[recum_col], errors='coerce')
    elif height_col:
        df['height_cm'] = pd.to_numeric(df[height_col], errors='coerce')
    else:
        df['height_cm'] = pd.NA
    return df[['SEQN', 'weight_kg', 'height_cm']]

# Đọc và gộp các chu kỳ
demo_files = sorted(glob.glob(os.path.join(RAW_NHANES_DIR, '*DEMO*.XPT')))
bmx_files = sorted(glob.glob(os.path.join(RAW_NHANES_DIR, '*BMX*.XPT')))

print(f"Tìm thấy {len(demo_files)} file DEMO và {len(bmx_files)} file BMX thô.")

combined_cycles = []
for demo_f in demo_files:
    prefix = os.path.basename(demo_f).split('_')[0]
    match_bmx = None
    for b in bmx_files:
        if os.path.basename(b).split('_')[0] == prefix:
            match_bmx = b
            break
            
    if not match_bmx:
        print(f"Cảnh báo: Không tìm thấy file BMX khớp cho DEMO {prefix}")
        continue
        
    # Bỏ qua các file 404 HTML lỗi có kích thước nhỏ (thường < 50KB)
    if os.path.getsize(demo_f) < 50000 or os.path.getsize(match_bmx) < 50000:
        continue
        
    print(f"Đang tiền xử lý chu kỳ: {prefix}...")
    try:
        # Sử dụng thư viện pyreadstat đã cài để đọc file XPT (Không bị lỗi byte string như pd.read_sas)
        import pyreadstat
        demo_df, _ = pyreadstat.read_xport(demo_f)
        bmx_df, _ = pyreadstat.read_xport(match_bmx)
        
        demo_norm = normalize_demo(demo_df)
        bmx_norm = normalize_bmx(bmx_df)
        
        merged = pd.merge(demo_norm, bmx_norm, on='SEQN', how='inner')
        merged = merged.dropna(subset=['weight_kg', 'height_cm'])
        combined_cycles.append(merged)
    except Exception as e:
        print(f"Lỗi chu kỳ {prefix}: {e}")

if combined_cycles:
    all_nhanes = pd.concat(combined_cycles, ignore_index=True)
    all_nhanes['age_months'] = pd.to_numeric(all_nhanes['age_months'], errors='coerce')
    # Lọc cho trẻ dưới 24 tháng
    nhanes_under24 = all_nhanes[all_nhanes['age_months'] < 24].reset_index(drop=True)
    print(f"--> Đã gộp thành công! Tổng số bản ghi trẻ em dưới 24 tháng: {len(nhanes_under24)}")
else:
    print("Không có dữ liệu chu kỳ nào được xử lý thành công.")
    nhanes_under24 = pd.DataFrame(columns=['SEQN', 'age_months', 'sex', 'weight_kg', 'height_cm'])


## Bước 5: Kiểm tra cấu trúc dữ liệu thô trước khi áp dụng Z-Score

In [ ]:
if nhanes_under24.empty:
    print("CẢNH BÁO: Tập dữ liệu nhanes_under24 hiện đang trống. Hãy kiểm tra xem file XPT đã tải lên đúng chưa.")
else:
    # Hiển thị 5 dòng đầu tiên
    print("5 bản ghi đầu tiên trong tập dữ liệu gộp:")
    display(nhanes_under24.head())

    # Kiểm tra thống kê cơ bản
    print("\nThống kê mô tả các thuộc tính số:")
    display(nhanes_under24.describe())

## Bước 6: Tính toán Z-Scores và Gán nhãn WHO cho từng bản ghi

In [ ]:
if nhanes_under24.empty:
    raise ValueError("Tập dữ liệu nhanes_under24 đang trống! Vui lòng kiểm tra lại xem Bước 4 đã tìm thấy và đọc được file *.XPT nào hay chưa.")

nhanes_under24['age_days'] = nhanes_under24['age_months'] * 30.4375
nhanes_under24['bmi'] = nhanes_under24['weight_kg'] / ((nhanes_under24['height_cm'] / 100.0) ** 2)

print("Bắt đầu đối chiếu bảng Excel WHO và tính Z-Scores... (Quá trình này có thể mất 1-2 phút)")

# Tính toán các nhãn WHO và chuyển trực tiếp thành DataFrame để tránh các lỗi logic của apply(axis=1) trên các phiên bản pandas khác nhau
who_list = []
for _, row in nhanes_under24.iterrows():
    res = classify_row(row)
    who_list.append(res.to_dict() if hasattr(res, 'to_dict') else res)

who_df = pd.DataFrame(who_list, index=nhanes_under24.index)

# Gán các cột vào dataframe chính
for col in ['who_metric', 'who_zscore', 'who_percentile', 'who_class']:
    nhanes_under24[col] = who_df[col]

print("Hoàn tất tính toán!")
print("Số lượng các nhãn phân loại thể trạng:")
print(nhanes_under24['who_class'].value_counts())

# Hiển thị mẫu dữ liệu sau khi dán nhãn
display(nhanes_under24.head())

## Bước 7: Trích xuất tập Feature cuối cùng và Xuất file CSV sạch

In [ ]:
if nhanes_under24.empty:
    raise ValueError("Tập dữ liệu rỗng, không thể xuất CSV.")

# Tạo cột mã hóa giới tính: male -> 1, female -> 0
nhanes_under24['gender_code'] = nhanes_under24['sex'].map({'male': 1, 'female': 0}).fillna(-1)

# Vì NHANES là dữ liệu cắt ngang (Cross-sectional), chúng ta đặt các biến lịch sử (tốc độ tăng trưởng) bằng 0
nhanes_under24['weight_delta_30d'] = 0.0
nhanes_under24['height_delta_30d'] = 0.0
nhanes_under24['measurement_interval_days'] = 0.0
nhanes_under24['head_circumference_cm'] = np.nan

# Lựa chọn các thuộc tính đưa vào mô hình học máy theo thiết kế hệ thống
final_features = nhanes_under24[[
    "age_days", "age_months", "gender_code", "weight_kg", "height_cm",
    "head_circumference_cm", "bmi", "who_zscore", "who_percentile",
    "weight_delta_30d", "height_delta_30d", "measurement_interval_days",
    "who_class"
]]

# Loại bỏ các dòng có nhãn bị lỗi hoặc chưa rõ
final_features = final_features[final_features['who_class'] != 'unknown'].dropna(subset=['who_class'])

print(f"Kích thước tập dữ liệu hoàn thiện: {final_features.shape}")

# Xuất file CSV lưu trên Google Drive làm đầu vào cho File 2
output_csv_path = os.path.join(PROCESSED_DIR, 'nhanes_under24_processed.csv')
final_features.to_csv(output_csv_path, index=False)
print(f"--> Đã xuất file CSV sạch tại: {output_csv_path}")